In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

coll = client.collections.create(
    "<COLLECTION_NAME>",
    vectorizer_config=Configure.Vectorizer.text2vec_openai(),
    properties=[
        Property(name="<PROP1>", data_type=DataType.TEXT),
        Property(name="<PROP2>", data_type=DataType.TEXT),
    ],
    multi_tenancy_config=Configure.multi_tenancy(True)
)

In [ ]:
from weaviate.classes.tenants import Tenant

# Add Tenant
coll = client.collections.get("<COLLECTION_NAME>")
coll.tenants.create([Tenant(name='<TENANT1>'), Tenant(name='<TENANT2>'), Tenant(name='<TENANT3>')])


In [ ]:

# How to Insert data in the tenants
# Define objects to insert into each tenant, with prop1 and prop2
data_objects = {
    "Tenant1": [
        {"content": "This is content for Tenant1 1", "book": "Content Tenant1 1"},
        {"content": "This is content for Tenant1 2", "book": "Content Tenant1 2"},
        {"content": "This is content for Tenant1 3", "book": "Content Tenant1 3"},
        {"content": "This is content for Tenant1 4", "book": "Content Tenant1 4"},
        {"content": "This is content for Tenant1 5", "book": "Content Tenant1 5"}
    ],
    "Tenant2": [
        {"content": "This is content for Tenant2 1", "book": "Content Tenant2 1"},
        {"content": "This is content for Tenant2 2", "book": "Content Tenant2 2"},
        {"content": "This is content for Tenant2 3", "book": "Content Tenant2 3"},
        {"content": "This is content for Tenant2 4", "book": "Content Tenant2 4"},
        {"content": "This is content for Tenant2 5", "book": "Content Tenant2 5"}
    ],
    "Tenant3": [
        {"content": "This is content for Tenant3 1", "book": "Content Tenant3 1"},
        {"content": "This is content for Tenant3 2", "book": "Content Tenant3 2"},
        {"content": "This is content for Tenant3 3", "book": "Content Tenant3 3"},
        {"content": "This is content for Tenant3 4", "book": "Content Tenant3 4"},
        {"content": "This is content for Tenant3 5", "book": "Content Tenant3 5"}
    ]
}
coll = client.collections.get("<COLLECTION_NAME>")
tenants = coll.tenants.get()
# Iterate through the tenants and insert data
for tenant_name, tenant_obj in tenants.items():
    tenant_collection = coll.with_tenant(tenant_name)  # Tenant-specific collection

# Iterate through the objects for each tenant and insert into properties
for data in data_objects[tenant_name]:
    tenant_collection.data.insert(
    properties={
        "content": data["content"],
        "book": data["book"]
        }
    )

In [ ]:
# How to update the tenants status
from weaviate.classes.tenants import Tenant, TenantActivityStatus

col = client.collections.get("<COLLECTION_NAME>")
col.tenants.update(tenants=[
    Tenant(
        name="Tenant1",
        activity_status=TenantActivityStatus.INACTIVE
    ),
    Tenant(
        name="Tenant2",
        activity_status=TenantActivityStatus.INACTIVE
    ),
    Tenant(
        name="Tenant3",
        activity_status=TenantActivityStatus.INACTIVE
    )
])

In [ ]:
# Retrieve Tenant Activity Status
import pandas as pd

col = client.collections.get("<COLLECTION_NAME>")
tenants = col.tenants.get()

# Extract tenant data into a list of dictionaries
tenant_data = []
for tenant_id, tenant in tenants.items():
    tenant_data.append({
        'Tenant ID': tenant_id,
        'Name': tenant.name,
        'Activity Status Internal': tenant.activityStatusInternal.name,
        'Activity Status': tenant.activityStatus.name
    })

# Convert the data to a pandas DataFrame
df = pd.DataFrame(tenant_data)
# Print the formatted table
print(df)


In [ ]:
import requests

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def get_schema():
    resp = requests.get(f"{CLUSTER_URL}/v1/schema", headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def check_multi_tenancy_status():
    """
    Check multi-tenancy status for all collections by reading the schema.
    Returns a dictionary mapping collection names to their MT status.
    """
    try:
        schema = get_schema()
        collections = schema.get('classes', [])
        
        if not collections:
            print("No collections found in schema.")
            return {}
        
        print("Collections in Weaviate:")
        mt_status = {}
        
        for collection in collections:
            collection_name = collection.get('class', 'Unknown')
            mt_config = collection.get('multiTenancyConfig', {})
            is_enabled = mt_config.get('enabled', False)
            
            mt_status[collection_name] = is_enabled
            status = "MT is enabled" if is_enabled else "MT is not enabled"
            print(f"- {collection_name}: {status}")
            
            # Optional: print additional MT details
            if is_enabled:
                auto_create = mt_config.get('autoTenantCreation', False)
                auto_activate = mt_config.get('autoTenantActivation', False)
                print(f"  └─ Auto-create: {auto_create}, Auto-activate: {auto_activate}")
        
        return mt_status
        
    except requests.exceptions.RequestException as e:
        print(f"Error retrieving schema: {e}")
        return {}
    except Exception as e:
        print(f"Unexpected error: {e}")
        return {}

print(check_multi_tenancy_status())